## Table of Contents
1. [Introduction](#introduction)
2. [Setting Up Your Environment](#setup)
3. [Constructing Your Target Universe](#universe)
4. [Retrieving Target Data](#data)
5. [Analyzing Data with Functions](#functions)
6. [Working with Multiple Data Items](#multiple)
7. [Handling Errors](#errors)
8. [Code Challenges](#challenges)

---

## Introduction {#introduction}

This tutorial is divided into sections teaching the fundamentals of writing and executing BQL queries in BQuant.

**Who this is for:** Quantitative Researchers and Analysts who want to start using BQL in their data analysis.

**What you'll learn:** How to write Python and BQL code to execute basic BQL queries and work with the output. You'll also learn where to find documentation and next steps in applying BQL to your analysis.

**Prerequisites:**
- Working knowledge of Python (but you don't need to be an expert)
- Comfort navigating Jupyter notebooks
- Familiarity with the BQL Overview concepts
- Basic understanding of financial markets and statistics

# BQL Basics Tutorial - Interactive
## Learn to Retrieve and Analyze Bloomberg Data in BQuant

Published: September 9, 2025 | Updated: January 12, 2026

This tutorial teaches the fundamentals of writing Python and BQL code to retrieve and analyze Bloomberg financial data in BQuant.

---

## Setting Up Your Environment {#setup}

Before you can start retrieving BQL data, you must set up your environment by:
1. Importing pandas for DataFrame handling
2. Importing the PyBQL library (bql)
3. Creating a BQL Service instance

### BQL Objects Overview

| BQL Object | PyBQL Property | Action | Example |
|---|---|---|---|
| Universe function | univ | Define target universe | univ.members() |
| Data item | data | Retrieve data points | data.px_last() |
| Function | func | Perform calculations | func.avg() |

In [ ]:
# Set up your environment
import pandas as pd
import bql

# Create a BQL Service instance
bq = bql.Service()

print("✓ Environment setup complete!")
print(f"BQL Service: {bq}")
print("Ready to retrieve Bloomberg data.")

### Your First BQL Query

Every BQL query requires two parts:
1. **Target Universe**: The securities or entities you want to analyze
2. **Target Data**: The specific data you want to retrieve

The BQL Calculation Engine processes your query and returns results as a DataFrame.

In [ ]:
# Example: Get the last price of Apple
universe = 'AAPL US Equity'
data_item = bq.data.px_last()

# Create and execute the request
request = bql.Request(universe, data_item)
response = bq.execute(request)

# Convert to DataFrame and display
data = response[0].df()
data

---

## Constructing Your Target Universe {#universe}

### Method 1: Using Bloomberg Identifiers

The simplest way to define your universe is with Bloomberg identifiers (tickers, ISINs, CUSIPs, etc.).

In [ ]:
# Single ID: Get company name
universe = 'AAPL US Equity'
data_item = bq.data.name()

request = bql.Request(universe, data_item)
response = bq.execute(request)
response[0].df()

In [ ]:
# Multiple IDs: Different identifier types
# Note: These are examples of different ID formats BQL supports
securities = [
    'AAPL US Equity',      # Ticker
    'MSFT US Equity',      # Ticker
    'GOOGL US Equity'      # Ticker
]

data_item = bq.data.px_last()

request = bql.Request(securities, data_item)
response = bq.execute(request)
response[0].df()

### Method 2: Using Universe Functions

Universe functions help you define universes dynamically based on relationships. Key functions include:
- `members()` - Get members of an index or portfolio
- `peers()` - Get peer companies  
- `bonds()` - Get bonds issued by a company
- `filter()` - Filter a universe based on conditions

In [ ]:
# Example: Get members of an index (current)
universe = bq.univ.members('INDU Index')
data_item = bq.data.name()

request = bql.Request(universe, data_item)
response = bq.execute(request)
print(f"Total Dow members: {len(response[0].df())}")
print("\nFirst 5 members:")
response[0].df().head()

In [ ]:
# Example: Historical data - members as of a specific date
universe = bq.univ.members('INDU Index', dates='2020-12-31')
data_item = bq.data.name()

request = bql.Request(universe, data_item)
response = bq.execute(request)
print("Dow members as of December 31, 2020:")
response[0].df().head()

### Method 3: Filtering Universes

Filter a universe based on conditions—either descriptive (sector, country) or market-based (volume, price).

In [ ]:
# Example: Filter Dow members by sector
universe = bq.univ.members('INDU Index')
data_item = bq.data.classification_name() 

# Create filtered universe for only Financials sector
filtered_universe = bq.univ.filter(universe, data_item == 'Financials')

request = bql.Request(filtered_universe, data_item)
response = bq.execute(request)
print("Dow Financials sector members:")
response[0].df()

---

## Retrieving Target Data {#data}

### Understanding Data Items

BQL data items represent values (e.g., price, earnings) plus associated metadata (currency, date, source). Data items vary widely based on data type:

**Categories:**
- Descriptive data (name, sector, country)
- Market data (prices, volume, yields)
- Fundamentals (earnings, revenue, cash flow)
- Ratings and credit data
- ESG metrics
- Economic statistics

In [ ]:
# Example 1: Last price (default parameters)
universe = 'AAPL US Equity'
data_item = bq.data.px_last()

request = bql.Request(universe, data_item)
response = bq.execute(request)
print("Current Apple stock price:")
response[0].df()

In [ ]:
# Example 2: Historical data with dates parameter
universe = 'AAPL US Equity'
data_item = bq.data.px_last(dates='2023-03-06')

request = bql.Request(universe, data_item)
response = bq.execute(request)
print("Apple price on March 6, 2023:")
response[0].df()

In [ ]:
# Example 3: Fundamentals data - quarterly EPS
universe = 'AAPL US Equity'
data_item = bq.data.is_eps(fa_period_type='q')

request = bql.Request(universe, data_item)
response = bq.execute(request)
print("Apple's latest quarterly EPS:")
response[0].df()

### Retrieving Time Series Data

Use the `range()` function to get data over a date range instead of a single date.

In [ ]:
# Example: Get 10-day price history
universe = 'AAPL US Equity'
data_item = bq.data.px_last(dates=bq.func.range('-10d', '0d'))

request = bql.Request(universe, data_item)
response = bq.execute(request)
print("Apple prices for the last 10 days:")
response[0].df()

---

## Analyzing Data with Functions {#functions}

BQL functions manipulate and transform data before returning results. Categories include:

| Function Type | Examples | Purpose |
|---|---|---|
| Data Handling | dropna(), fill() | Remove or fill missing values |
| Statistical | max(), min(), avg(), median(), std() | Calculate statistics |
| Time Series | net_chg(), pct_chg(), diff(), pct_diff() | Calculate changes |
| Grouping | group(), groupavg(), groupmax() | Aggregate or compare data |

In [ ]:
# Example 1: Statistical function - max price in last 10 days
universe = 'AAPL US Equity'
data_item = bq.data.px_open(dates=bq.func.range('-9d', '0d')).max()

request = bql.Request(universe, data_item)
response = bq.execute(request)
print("Highest opening price for Apple in last 10 days:")
response[0].df()

In [ ]:
# Example 2: Time series function - percent change over 2 weeks
universe = 'AAPL US Equity'
data_item = bq.data.px_last(dates=bq.func.range('-2w', '0d')).pct_chg()

request = bql.Request(universe, data_item)
response = bq.execute(request)
print("Apple percent change over last 2 weeks:")
response[0].df()

In [ ]:
# Example 3: Grouping - average volume for multiple stocks
universe = ['IBM US Equity', 'AAPL US Equity', 'MSFT US Equity']
data_item = bq.data.px_volume(dates=bq.func.range('-29d', '0d')).avg()

request = bql.Request(universe, data_item)
response = bq.execute(request)
print("30-day average volume for tech stocks:")
response[0].df()

---

## Working with Multiple Data Items {#multiple}

You can construct queries with multiple data items using a dictionary. Use `['value']` to extract only the value without metadata, then use pandas to combine results.

In [ ]:
# Example: Create a simple comp sheet
universe = 'AAPL US Equity'

# Create a dictionary of multiple data items with value-only output
data_items = {
    'Last Price': bq.data.px_last()['value'],
    'Volume': bq.data.px_volume()['value'],
    'Market Cap': bq.data.cur_mkt_cap()['value']
}

request = bql.Request(universe, data_items)
response = bq.execute(request)

# Combine results using pandas concat
data = pd.concat([item.df() for item in response], axis=1)
print("Apple Company Summary:")
data

---

## Handling Errors and Debugging {#errors}

### Three Types of BQL Errors

| Error Type | Cause | Impact | Example |
|---|---|---|---|
| Query Level | Syntax errors, missing arguments | Entire query fails | Missing data_item in Request |
| Item Level | Data type mismatches | Single data item fails | avg(string_data, numeric_data) |
| Row Level | Unit/currency mismatches | Some rows fail | Different currencies in calculation |

---

## Code Challenges {#challenges}

### Challenge 1: Relative Valuation
Write a BQL query that uses `peers()` and `is_eps()` to retrieve the earnings per share of Apple Inc.'s peer companies.

### Challenge 2: Comp Sheet Creation
Create a scalable comp sheet for FAANG companies with these data items:
- `name()` - Company name
- `px_last()` - Current price
- `cur_mkt_cap()` - Market capitalization
- `curr_entp_val()` - Enterprise value
- `sales_rev_turn()` - Revenue

### Challenge 3: ESG Analysis
Write a query to get the count of green bonds issued by S&P 500 member companies, grouped by BICS sector, sorted from highest to lowest count.

### Challenge 4: Technical Analysis
Find the top 10 dates over the last 5 years when a security (e.g., Vodafone) had the largest one-day percent price moves.

---

## Summary and Next Steps

**You've learned:**
✓ How to set up your BQL environment
✓ Three ways to construct target universes (IDs, functions, filtering)
✓ How to retrieve various types of data with parameters
✓ How to apply functions for analysis, aggregation, and transformation
✓ How to work with multiple data items and comp sheets
✓ How to identify and fix different error types

**Next steps:**
1. **Explore more data items** - Visit the BQuant Help Center to discover 19,597+ available data items
2. **Try the code challenges** - Apply what you've learned with real-world scenarios
3. **Read specialized tutorials**:
   - BQL for Equities Tutorial (equity-specific analysis)
   - Intro to Fixed Income with BQL (bond analysis)
   - Visualizations Using Plotly (data visualization)
4. **Consult the PyBQL API Reference** for detailed documentation on all functions

**Key Resources:**
- BQuant Help Center: Complete documentation for data items, functions, and more
- Bloomberg Terminal: Use FLDS <GO> to search for specific data items
- Tab completion in BQuant: Use Python's help() and __doc__() to explore available objects

Happy querying! 🎉